In [ ]:
# Library imports
import torch
import math
import random
import json
import tqdm

import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from dotenv import dotenv_values

from src.claim_extraction.config import DOTENV_PATH
from src.claim_extraction.extractor import extract_claims

In [2]:

env = dotenv_values(DOTENV_PATH)

In [3]:
# Determine what device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(device)

    print(f'CUDA on {device_name}')
else:
    print(f'CPU')

CUDA on NVIDIA GeForce RTX 3060 Laptop GPU


In [4]:
# Check if everything's cleaned
allocated = torch.cuda.memory_allocated() / (1024 * 1024)
reserved = torch.cuda.memory_reserved() / (1024 * 1024)

print(f'Allocated: {allocated:.1f} MB; Reserved: {reserved:.1f} MB')

Allocated: 0.0 MB; Reserved: 0.0 MB


In [5]:
NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' # https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2' # https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

NLTK_MODEL = 'tokenizers/punkt/english.pickle'

# Standard generation parameters used by all runs
CLAIM_MODEL_NAME = env.get('CLAIM_MODEL_1', 'Qwen/Qwen3.5-4B')
CLAIM_MODEL_TEMPERATURE = 0.1
CLAIM_MODEL_MAX_NEW_TOKENS = 1536
CLAIM_MODEL_BACKEND = 'local'
CLAIM_MODEL_VERBOSE = True

In [6]:
#nltk.download('punkt_tab')

In [7]:
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)

embed_model = SentenceTransformer(EMBED_MODEL).to(device)

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
dataset = json.load(open('datasets/ContraDoc/ContraDoc.json', 'r'))

# Label examples
for example in dataset['pos'].values():
	example['label'] = 1.0 

for example in dataset['neg'].values():
	example['label'] = 0.0 

# Flatten list
all_examples = [ *dataset['pos'].values(), *dataset['neg'].values() ]

print(f'Dataset loaded with {len(dataset["pos"])} positive and {len(dataset["neg"])} negative examples.')

Dataset loaded with 449 positive and 442 negative examples.


In [9]:

def get_closest_sentence_pairs(sentences, embeddings, top_k=None):
    """
    Given a list of sentences and their embeddings, return sentence pairs with closest embeddings.
    
    Args:
        sentences: list of sentence strings
        embeddings: numpy array of embeddings for each sentence
        top_k: number of closest pairs to return
    
    Returns:
        list of tuples (sentence_a, sentence_b, similarity_score)
    """
    
    # Compute pairwise cosine similarity
    similarity_matrix = cosine_similarity(embeddings)
    
    for i in range(len(sentences)):
        similarity_matrix[i][i] = -1.0  # Exclude self-similarity by setting diagonal to -1

    pairs = [ (i, np.argmax(similarity_matrix[i])) for i in range(len(sentences)) ]
    
    # Sort by similarity score (descending) and get top-k
    if top_k is not None:
        pairs.sort(key=lambda x: similarity_matrix[x[0]][x[1]], reverse=True)
        pairs = pairs[:top_k]
        
    # Return pairs with sentences and similarity scores
    return [(sentences[i], sentences[j], similarity_matrix[i][j]) for i, j in pairs]

def predict(premise, hypothesis):
    input = nli_tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)

    output = nli_model(input["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]

    return {name: float(pred) for pred, name in zip(prediction, label_names)}
    
   
def detect(document):
    claims = extract_claims(
        text=document,
        model_name=CLAIM_MODEL_NAME,
        backend=CLAIM_MODEL_BACKEND,
        temperature=CLAIM_MODEL_TEMPERATURE,
        max_new_tokens=CLAIM_MODEL_MAX_NEW_TOKENS,
        verbose=CLAIM_MODEL_VERBOSE,
    )

    print(f'Extracted {len(claims)} claims:')
    print(claims)

    embeddings = embed_model.encode(claims)
    pairs = get_closest_sentence_pairs(claims, embeddings)

    p_contra_max = 0.0
    evidence = []

    for a, b, similarity in tqdm.tqdm(pairs):
        prediction = predict(a, b)
        p_contra = prediction['contradiction']
        p_contra_max = max(p_contra_max, p_contra)

        print(f'Comparing:')
        print(f'  {a}')
        print(f'  {b}')
        print(f'  Similarity: {similarity:.4f}')
        print(f'  Prediction: {p_contra:.4f}')
        print()

        if p_contra > 0.75:
            evidence.append((a, b, p_contra))

    return p_contra_max, evidence

In [10]:
premise = "The sky is red"
hypothesis = "The sky is blue"

print(predict(premise, hypothesis))

{'entailment': 8.13603401184082e-05, 'neutral': 0.0005106925964355469, 'contradiction': 0.99951171875}


In [11]:
pos = False					# Test example from positive or negative set
example_id = None			# Specific example ID to test, or None for random

# 3489738232_6
# story_train_3585

examples = dataset['pos' if pos else 'neg']

if example_id is not None:
	example = examples[example_id]
else:
	examples = list(examples.values())
	example = examples[random.randint(0, len(examples) - 1)]

print(f'Detecting contradictions in example {example["unique id"]}')
print()

print(example['text'])
print()

if pos:
	print(f'Evidence: {example["evidence"]}')
	print(f'Ref sentences: {example.get("ref sentences", [])}')

print()

p_contra, evidence = detect(example['text'])
p_contra, evidence

Detecting contradictions in example wiki_train_29316

 = HMS Swordfish ( 1916 ) = 
 
 HMS Swordfish was an experimental submarine built for the Royal Navy before the First World War to meet the Navy 's goal of an " overseas " submarine capable of 20 knots ( 37 km / h ; 23 mph ) on the surface. Diesel engines of the period were unreliable and not very powerful so steam turbines were proposed instead to meet the RN 's requirement. Swordfish proved to be slower than designed and unstable while surfacing , and consequently she was modified as an anti - submarine patrol vessel in 1917. She was paid off before the end of the war and sold for scrapping in 1922. 
 
 = = Design = = 
 
 HMS Swordfish was developed to meet a requirement of Royal Navy 's Submarine Committee for a large submarine capable of operating with the fleet at a surfaced speed of 20 knots. Most of the earlier British submarines had been single - hulled vessels built by Vickers , and the Navy was interested in evaluating oth

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

c:\Users\erwin\miniconda3\envs\contradetect\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\erwin\.cache\huggingface\hub\models--Qwen--Qwen3.5-0.8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

...

 96%|█████████▌| 45/47 [00:03<00:00, 16.94it/s]

Comparing:
  = Captained by Commander Geoffrey Layton, her post-completion trials lasted for five months as she was used to evaluate steam power for submarine use.
  = These problems, coupled with the fact that she was too slow to work with the fleet as originally envisioned, meant that she was impossible to make into an effective warship, and she was laid up after her trials.
  Similarity: 0.4762
  Prediction: 0.0008

Comparing:
  = Much was learned about the operation of steam submarines, which helped the subsequent design of the steam-powered K-class fleet submarines.
  = Most of the earlier British submarines had been single-hulled vessels built by Vickers, and the Navy was interested in evaluating other designs.
  Similarity: 0.5504
  Prediction: 0.0001

Comparing:
  = She proved to be very unstable while surfacing, presumably because she could not pump the water out of her controlled free-flooding spaces quickly enough in the upper part of her double hull.
  = The upper portion o

100%|██████████| 47/47 [00:03<00:00, 14.08it/s]

Comparing:
  = In July 1917 S1 reverted to her original name and was converted to a surface patrol vessel between 27 June 1917
  = HMS Swordfish was slower than designed and unstable while surfacing, so she was modified as an anti-submarine patrol vessel in 1917.
  Similarity: 0.5524
  Prediction: 0.0019



(0.9970703125,
 [('= Swordfish had two Parsons geared impulse-reaction steam turbine sets, each driving one of the two propeller shafts.',
   "= HMS Swordfish was built with unreliable diesel engines and proposed steam turbines to meet the Royal Navy's requirement.",
   0.97412109375),
  ('= They were designed to produce a total of 4,000 shaft horsepower (3,000 kW) at a working pressure of 250 psi (1,700 kPa; 18 kgf/cm2) which used a superheater to increase the working temperature by 100°F (38°C).',
   '= She was fitted with two electric motors which had a combined output of 1,400 bhp (1,000 kW).',
   0.99609375),
  ('= She was fitted with two electric motors which had a combined output of 1,400 bhp (1,000 kW).',
   '= They were designed to produce a total of 4,000 shaft horsepower (3,000 kW) at a working pressure of 250 psi (1,700 kPa; 18 kgf/cm2) which used a superheater to increase the working temperature by 100°F (38°C).',
   0.9755859375),
  ('= Swordfish could carry 102 long tons

In [ ]:
# Evaluation loop

results = []
for example in tqdm.tqdm(all_examples):
	unique_id = example["unique id"]
	text = example["text"]
	y_true = example["label"]        # 0 or 1
	doc_type = example["doc_type"]   # story/wiki/news

	p_pred, evidence = (random.random(), []) #predict(text)

	results.append({
		"unique_id": unique_id,
		"p_pred": p_pred,
		"y_true": y_true,
		"doc_type": doc_type
	})

100%|██████████| 891/891 [00:00<00:00, 68543.43it/s]


In [ ]:
# Write output file
with open('./data/results.json', "w") as f:
    json.dump(results, f, indent=2)